# install unsloth 

In [1]:
%%capture
!pip install unsloth

## load the model in 4bit QLORA

In [2]:
import os, sys, contextlib, torch, warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["UNSLOTH_SILENT"] = "1"

print("Loading Unquantized Qwen 0.6B...")

with contextlib.redirect_stdout(open(os.devnull, 'w')):
    with contextlib.redirect_stderr(open(os.devnull, 'w')):
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
            max_seq_length=2048,
            dtype=torch.float16,
            load_in_4bit=True
        )

print("✅ Model loaded successfully.")

Loading Unquantized Qwen 0.6B...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded successfully.


In [3]:
print("Injecting LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print("LoRA adapters injected successfully.")

Injecting LoRA adapters...
LoRA adapters injected successfully.


In [4]:
from datasets import (
    load_dataset,
    DatasetDict,
    concatenate_datasets,
)


DATA_PATH = (
    "/kaggle/input/datasets/atomstack001/"
    "bioreasoning-sft-trial-data/"
    "sft_rejection_sampled_train_phase_one.jsonl"
)

USER_TOKEN = "<｜User｜>"
ASSISTANT_TOKEN = "<｜Assistant｜>"


# =========================================================
# DeepSeek training-text formatter
# =========================================================

def build_deepseek_training_text(messages):
    """
    Build DeepSeek-R1-Distill SFT text manually.

    The default DeepSeek tokenizer template removes content
    before </think>. This formatter preserves the complete
    reasoning trace.
    """

    system_parts = []
    user_parts = []
    assistant_parts = []

    for message in messages:
        role = message["role"]
        content = message["content"].strip()

        if role == "system":
            system_parts.append(content)

        elif role == "user":
            user_parts.append(content)

        elif role == "assistant":
            assistant_parts.append(content)

        else:
            raise ValueError(
                f"Unsupported message role: {role}"
            )

    if not user_parts:
        raise ValueError(
            "Conversation has no user message."
        )

    if len(assistant_parts) != 1:
        raise ValueError(
            "Expected exactly one assistant response, "
            f"found {len(assistant_parts)}."
        )

    # DeepSeek recommends avoiding a separate system role.
    # Preserve the system instructions by merging them into
    # the same user turn.
    merged_user_prompt = "\n\n".join(
        system_parts + user_parts
    ).strip()

    assistant_response = assistant_parts[0].strip()

    if "</think>" not in assistant_response:
        raise ValueError(
            "Assistant response is missing </think>."
        )

    # Force every reasoning response to begin with <think>.
    if not assistant_response.startswith("<think>"):
        assistant_response = (
            "<think>\n"
            + assistant_response
        )

    bos_token = tokenizer.bos_token or ""
    eos_token = tokenizer.eos_token or ""

    formatted_text = (
        f"{bos_token}"
        f"{USER_TOKEN}"
        f"{merged_user_prompt}"
        f"{ASSISTANT_TOKEN}"
        f"{assistant_response}"
        f"{eos_token}"
    )

    return formatted_text


def format_deepseek_batch(examples):
    texts = []

    for messages in examples["messages"]:
        formatted_text = build_deepseek_training_text(
            messages
        )
        texts.append(formatted_text)

    return {"text": texts}


# =========================================================
# Stratified subset helper
# =========================================================

def create_stratified_subset(
    source_dataset,
    label_counts,
    seed,
):
    subsets = []

    for label, requested_count in label_counts.items():
        print(
            f"Selecting {requested_count} "
            f"examples for label '{label}'..."
        )

        label_dataset = source_dataset.filter(
            lambda example: example["label"] == label,
            desc=f"Filtering {label}",
        )

        if requested_count > len(label_dataset):
            raise ValueError(
                f"Requested {requested_count} '{label}' rows, "
                f"but only {len(label_dataset)} are available."
            )

        selected_dataset = (
            label_dataset
            .shuffle(seed=seed)
            .select(range(requested_count))
        )

        subsets.append(selected_dataset)

        print(
            f"Selected {len(selected_dataset)} "
            f"'{label}' examples."
        )

    combined_dataset = concatenate_datasets(
        subsets
    ).shuffle(seed=seed)

    return combined_dataset


# =========================================================
# Load complete JSONL
# =========================================================

print("Loading local SFT dataset...")

raw_dataset = load_dataset(
    "json",
    data_files=DATA_PATH,
    split="train",
)

print(f"Total rows loaded: {len(raw_dataset)}")


print("Reading original train split...")

full_train_dataset = raw_dataset.filter(
    lambda example: example["split"] == "train",
    desc="Reading full training split",
)


print("Reading original validation split...")

full_val_dataset = raw_dataset.filter(
    lambda example: example["split"] == "val",
    desc="Reading full validation split",
)


print(
    f"Full training rows available: "
    f"{len(full_train_dataset)}"
)

print(
    f"Full validation rows available: "
    f"{len(full_val_dataset)}"
)


# =========================================================
# FULL DATASET OPTION
#
# Uncomment these two lines to train on all 6,739 rows
# and validate on all 749 rows.
# =========================================================

# train_dataset = full_train_dataset
# val_dataset = full_val_dataset


# =========================================================
# REDUCED STRATIFIED DATASET — ACTIVE
#
# 2,000 training examples
# 220 validation examples
# =========================================================

print("Creating reduced stratified training set...")

train_dataset = create_stratified_subset(
    full_train_dataset,
    label_counts={
        "none": 1112,
        "up": 603,
        "down": 285,
    },
    seed=42,
)


print("Creating reduced stratified validation set...")

val_dataset = create_stratified_subset(
    full_val_dataset,
    label_counts={
        "none": 122,
        "up": 66,
        "down": 32,
    },
    seed=43,
)


assert len(train_dataset) == 2000
assert len(val_dataset) == 220

print(f"Selected training rows: {len(train_dataset)}")
print(f"Selected validation rows: {len(val_dataset)}")


# =========================================================
# Apply DeepSeek formatting
# =========================================================

print("Formatting training examples...")

train_dataset = train_dataset.map(
    format_deepseek_batch,
    batched=True,
    desc="Formatting training examples",
    load_from_cache_file=False,
)


print("Formatting validation examples...")

val_dataset = val_dataset.map(
    format_deepseek_batch,
    batched=True,
    desc="Formatting validation examples",
    load_from_cache_file=False,
)


dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
})


# =========================================================
# Validate rendered training text
# =========================================================

print("Validating formatted examples...")

for split_name in ["train", "validation"]:
    sample = dataset[split_name][0]["text"]

    assert USER_TOKEN in sample
    assert ASSISTANT_TOKEN in sample
    assert "Your input fields are:" in sample
    assert "[[ ## pert ## ]]" in sample
    assert "[[ ## gene ## ]]" in sample

    assistant_start = sample.rindex(
        ASSISTANT_TOKEN
    )

    think_start = sample.index(
        "<think>",
        assistant_start,
    )

    think_end = sample.index(
        "</think>",
        think_start,
    )

    label_start = sample.index(
        "[[ ## label ## ]]",
        think_end,
    )

    completed_start = sample.index(
        "[[ ## completed ## ]]",
        label_start,
    )

    assert (
        assistant_start
        < think_start
        < think_end
        < label_start
        < completed_start
    )

    assistant_section = sample[assistant_start:]

    assert assistant_section.count("<think>") == 1
    assert assistant_section.count("</think>") == 1
    assert (
        assistant_section.count(
            "[[ ## label ## ]]"
        )
        == 1
    )
    assert (
        assistant_section.count(
            "[[ ## completed ## ]]"
        )
        == 1
    )

    print(
        f"✅ {split_name} example validated."
    )


print("Formatted-data validation passed.")
print("Training rows:", len(dataset["train"]))
print("Validation rows:", len(dataset["validation"]))
print("Training columns:", dataset["train"].column_names)
print(
    "Validation columns:",
    dataset["validation"].column_names,
)


# =========================================================
# Print complete rendered examples
# =========================================================

print("\n" + "=" * 80)
print("FULL TRAINING TEXT SENT TO THE TRAINER")
print("=" * 80)
print(dataset["train"][0]["text"])
print("=" * 80)


print("\n" + "=" * 80)
print("FULL VALIDATION TEXT SENT TO THE TRAINER")
print("=" * 80)
print(dataset["validation"][0]["text"])
print("=" * 80)

Loading local SFT dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Total rows loaded: 7488
Reading original train split...


Reading full training split:   0%|          | 0/7488 [00:00<?, ? examples/s]

Reading original validation split...


...

Selected 32 'down' examples.
Selected training rows: 2000
Selected validation rows: 220
Formatting training examples...


Formatting training examples:   0%|          | 0/2000 [00:00<?, ? examples/s]

Formatting validation examples...


Formatting validation examples:   0%|          | 0/220 [00:00<?, ? examples/s]

Validating formatted examples...
✅ train example validated.
✅ validation example validated.
Formatted-data validation passed.
Training rows: 2000
Validation rows: 220
Training columns: ['messages', 'row_id', 'label', 'accepted_trial', 'split', 'text']
Validation columns: ['messages', 'row_id', 'label', 'accepted_trial', 'split', 'text']

FULL TRAINING TEXT SENT TO THE TRAINER
<｜begin▁of▁sentence｜><｜User｜>Your input fields are:
1. `pert` (str): The knocked-down perturbation gene
2. `gene` (str): The target gene to predict
Your output fields are:
1. `label` (str): Final label: exactly 'up', 'down', or 'none'

All interactions will be structured in the following way, with the appropriate values filled.
[[ ## pert ## ]]
{pert}
[[ ## gene ## ]]
{gene}
[[ ## label ## ]]
{label}
[[ ## completed ## ]]

In adhering to this structure, your objective is: 
You are an expert molecular and cellular biology expert analyzing Perturb-seq data from mouse bone-marrow-derived macrophages (BMDMs) stimulate

In [5]:
from trl import SFTTrainer, SFTConfig

print("Setting up SFT Trainer...")
print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")

assert len(dataset["train"]) == 2000
assert len(dataset["validation"]) == 220
assert "text" in dataset["train"].column_names
assert "text" in dataset["validation"].column_names

training_config = SFTConfig(
    output_dir="/kaggle/working/llama_8b_adapter",

    dataset_text_field="text",
    max_length=2048,
    packing=False,

    # Training schedule
    num_train_epochs=2,

    # Effective batch size: 1 × 4 = 4
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # Validation
    per_device_eval_batch_size=1,
    eval_strategy="epoch",

    # Optimization
    learning_rate=1e-4,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.0,

    # Logging
    logging_strategy="steps",
    logging_steps=100,
    logging_first_step=True,

    # Save after each epoch
    save_strategy="epoch",
    save_total_limit=2,

    # Restore the epoch with the lowest validation loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Tesla T4 settings
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,

    # Reproducibility
    seed=42,
    data_seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_config,
)

print("SFT Trainer ready.")
print("Training epochs: 2")
print("Expected optimizer steps: 1,000")
print("Validation: after every epoch")
print("Checkpointing: after every epoch")
print("Logging: every 100 optimizer steps")

Setting up SFT Trainer...
Training examples: 2000
Validation examples: 220


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/220 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
SFT Trainer ready.
Training epochs: 2
Expected optimizer steps: 1,000
Validation: after every epoch
Checkpointing: after every epoch
Logging: every 100 optimizer steps


In [6]:
# --- KEY METRICS LOGGING (BEFORE TRAINING) ---
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print("\n" + "="*50)
print(f"GPU: {gpu_stats.name} ({max_memory} GB Total)")
print(f"VRAM Reserved Before Training: {start_gpu_memory} GB")
print("="*50 + "\n")

# --- START TRAINING ---
print("🚀 Starting Training! Watch the loss go down! 📉")
trainer_stats = trainer.train()

# --- KEY METRICS LOGGING (AFTER TRAINING) ---
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)

print("\n" + "="*50)
print("🎯 TRAINING COMPLETE!")
print("="*50)
print(f"⏱️ Time taken:        {trainer_stats.metrics['train_runtime']/60:.2f} minutes")
print(f"🔥 Peak VRAM Used:    {used_memory} GB")
print(f"💾 VRAM for LoRA:     {used_memory_for_lora} GB")
print("="*50 + "\n")


save_path = "/kaggle/working/llama_8b_adapter/final_adapter"

print("\n" + "=" * 50)
print("Saving LoRA adapter...")
print(f"Save path: {save_path}")
print("=" * 50)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("\n" + "=" * 50)
print("LoRA adapter saved successfully!")
print(f"Saved to: {save_path}")
print("=" * 50)


GPU: Tesla T4 (14.562 GB Total)
VRAM Reserved Before Training: 5.787 GB

🚀 Starting Training! Watch the loss go down! 📉
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.465483,0.453070
2,0.383358,0.434797



🎯 TRAINING COMPLETE!
⏱️ Time taken:        273.34 minutes
🔥 Peak VRAM Used:    8.26 GB
💾 VRAM for LoRA:     2.473 GB


Saving LoRA adapter...
Save path: /kaggle/working/llama_8b_adapter/final_adapter

LoRA adapter saved successfully!
Saved to: /kaggle/working/llama_8b_adapter/final_adapter
